# F-S-N-SUPP: Reconstruction Error vs. Sensor SNR

# Imports

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np


def find_repo_root(start=Path.cwd()):
    for path in [start.resolve(), *start.resolve().parents]:
        if (path / "src" / "cs_priors").exists():
            return path
    raise RuntimeError("Could not find repository root")


REPO_ROOT = find_repo_root()
sys.path[:0] = [
    str(REPO_ROOT / "src"),
    str(REPO_ROOT / "notebooks" / "benchmarks_v2" / "functions"),
]

from figures import save_pdf
from single_frequency_benchmarks import (
    make_initial_solution,
    make_simulation,
    plot_metric,
    run_snr_benchmark,
    solve_methods,
    plot_source_amplitudes
)

from single_frequency_config import (
    FULL_RING_SPAN,
    LASSO_ALPHA,
    LASSO_MAX_ITER,
    METHOD_COLORS,
    METHOD_LINESTYLES,
    METHOD_LABELS,
    PHASE_SEEDS,
    base_sim_kwargs,
    sector_span_deg,
)

from sanity import run_sanity_check


def format_snr_axis(ax, results_df, ticks_to_label=None):
    snr_ticks = (
        results_df[["sensor_snr_db", "sensor_snr_label"]]
        .drop_duplicates()
        .sort_values("sensor_snr_db", ascending=False)
    )

    ax.set_xticks(snr_ticks["sensor_snr_db"])

    if ticks_to_label is None:
        ax.set_xticklabels(snr_ticks["sensor_snr_label"].str.replace(" dB", ""))
        ax.tick_params(axis="x", labelrotation=45)
    else:
        ax.set_xticklabels([
            label.replace(" dB", "") if label in ticks_to_label else ""
            for label in snr_ticks["sensor_snr_label"]
        ])
        ax.tick_params(axis="x", labelrotation=0)

    ax.invert_xaxis()


def plot_snr_metric(results_df, method_order, title, save_path, y_col):
    plot_df = results_df[results_df["method"].isin(method_order)].copy()

    fig, ax, summary_df = plot_metric(
    plot_df,
    x_col="sensor_snr_db",
    y_col=y_col,
    method_order=method_order,
    colors=METHOD_COLORS,
    linestyles=METHOD_LINESTYLES,
    xlabel="Sensor SNR (dB)",
    title=title,
    xscale="log",
    yscale="linear",
)


    format_snr_axis(ax, plot_df)
    save_pdf(fig, save_path)
    plt.show()

    return summary_df

# Simulation Parameters

In [ ]:
TAG = "F-S-N-SUPP"
FIGURE_DIR = REPO_ROOT / "results" / "benchmarks" / "v2" / TAG
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

NUM_MICS = 6
NUM_SOURCES = 10
SECTOR_SPAN_DEG = sector_span_deg(NUM_SOURCES)

SNR_DB = [None, 40, 30, 25, 20, 15, 13, 11, 9, 8, 7, 6, 5,4,3,2,1]

RIDGE_ALPHA = 1e-3
SPARSE_GROUPING = "none"
SPARSE_PRECISION = 1.0
SPARSE_EPS = 0.005

MP_METHODS_TO_PLOT = ["LASSO from MP", "Sparse Prior from MP", "MP", "LASSO from Ridge", "Sparse Prior from Ridge", "Ridge"]
RIDGE_METHODS_TO_PLOT = ["Sparse Prior from Ridge", "Ridge", "Sparse Prior from MP", "MP"]

RECOVERY_METHODS = {
    "MP": {
        "solver": "initializer",
        "initializer": "mp",
    },
    "LASSO from MP": {
        "solver": "lasso",
        "initializer": "mp",
        "lasso_alpha": LASSO_ALPHA,
        "lasso_max_iter": LASSO_MAX_ITER,
    },
    "Sparse Prior from MP": {
        "solver": "sparse_prior",
        "initializer": "mp",
        "sparse_grouping": SPARSE_GROUPING,
        "sparse_precision": SPARSE_PRECISION,
        "sparse_eps": SPARSE_EPS,
    },
    "Ridge": {
        "solver": "initializer",
        "initializer": "ridge",
        "ridge_alpha": RIDGE_ALPHA,
    },
    "LASSO from Ridge": {
        "solver": "lasso",
        "initializer": "ridge",
        "ridge_alpha": RIDGE_ALPHA,
        "lasso_alpha": LASSO_ALPHA,
        "lasso_max_iter": LASSO_MAX_ITER,
    },
    "Sparse Prior from Ridge": {
        "solver": "sparse_prior",
        "initializer": "ridge",
        "ridge_alpha": RIDGE_ALPHA,
        "sparse_grouping": SPARSE_GROUPING,
        "sparse_precision": SPARSE_PRECISION,
        "sparse_eps": SPARSE_EPS,
    },
}

SIM_KWARGS = base_sim_kwargs(
    num_mics=NUM_MICS,
    num_sources=NUM_SOURCES,
    mic_angle_span=FULL_RING_SPAN,
    source_angle_span_deg=SECTOR_SPAN_DEG,
)

# Example solution

In [ ]:
EXAMPLE_SNR_DB = 20

sim = make_simulation(
    SECTOR_SPAN_DEG,
    PHASE_SEEDS[0],
    sensor_snr_db=EXAMPLE_SNR_DB,
    **SIM_KWARGS,
)



solutions = solve_methods(
    sim,
    lasso_alpha=LASSO_ALPHA,
    lasso_max_iter=LASSO_MAX_ITER,
)

ridge_solution = make_initial_solution(
    sim.A,
    sim.Y,
    method="ridge",
    ridge_alpha=RIDGE_ALPHA,
)
solutions = {
    **solutions,
    "Ridge": ridge_solution,
}


run_sanity_check(
    sim,
    solutions=solutions,
    save_path=FIGURE_DIR / "simulation_geometry_num_mics_check.pdf",
)



solutions = {
    "MP": make_initial_solution(sim.A, sim.Y, method="mp"),
    "Ridge": make_initial_solution(sim.A, sim.Y, method="ridge", ridge_alpha=RIDGE_ALPHA),
}

fig, axes = plot_source_amplitudes(
    sim,
    solutions,
    labels=METHOD_LABELS,
    title=f"Example of predicted amplitudes for {len(sim.mics)} mics and {len(sim.sources)} sources with {EXAMPLE_SNR_DB} dB SNR",
)
save_pdf(fig, FIGURE_DIR / "source_amplitudes_example_mp_ridge.pdf")
plt.show()

# Simulating the results

In [ ]:
results_df = run_snr_benchmark(
    SNR_DB,
    PHASE_SEEDS,
    SECTOR_SPAN_DEG,
    RECOVERY_METHODS,
    sim_kwargs=SIM_KWARGS,
)

results_df = results_df.sort_values(
    ["sensor_snr_db", "phase_seed", "method"]
).reset_index(drop=True)

results_df.head()

# 1. Plotting performance vs SNR for LASSO, and Sparse prior initialized with MP, and for MP itself

In [ ]:
mp_summary_df = plot_snr_metric(
    results_df,
    MP_METHODS_TO_PLOT,
    f"Error vs SNR for {NUM_MICS} mics and {NUM_SOURCES} sources, ridge "+ r"$\alpha=$" +  f"{RIDGE_ALPHA}",
    FIGURE_DIR / "relative_complex_error_vs_sensor_snr.pdf",
    y_col="relative_complex_error",
)

mp_summary_df.head()

## 1. Plotting performance vs SNR for Ridge with $\alpha=10^{-3}$. Also LASSO, and Sparse prior, both initialized with Ridge with $\alpha=10^{-3}$

In [ ]:
from single_frequency_benchmarks import relabel_legend

# ridge_methods_with_mp = [*RIDGE_METHODS_TO_PLOT, "MP"]
# plot_df = results_df[results_df["method"].isin(ridge_methods_with_mp)].copy()

fig, ax, ridge_summary_df = plot_metric(
    results_df[results_df["method"].isin(RIDGE_METHODS_TO_PLOT)],
    x_col="sensor_snr_db",
    method_order=RIDGE_METHODS_TO_PLOT,
    colors=METHOD_COLORS,
    linestyles=METHOD_LINESTYLES,
    xlabel="Sensor SNR (dB)",
    title=f"Error vs SNR for {NUM_MICS} mics and {NUM_SOURCES} sources, ridge "
    + r"$\alpha=$"
    + f"{RIDGE_ALPHA}",
    xscale="log",
    yscale="linear",
)

format_snr_axis(ax, results_df)
relabel_legend(ax, METHOD_LABELS)
# ax.set_xlim(110,9)
# ax.set_ylim(0, 2)

save_pdf(fig, FIGURE_DIR / "relative_complex_error_vs_sensor_snr_ridge_initialization.pdf")
plt.show()

ridge_summary_df.head()




In [ ]:
mp_summary_df = plot_snr_metric(
    results_df,
    MP_METHODS_TO_PLOT,
    f"F1 vs SNR for {NUM_MICS} mics and {NUM_SOURCES} sources, MP initialization",
    FIGURE_DIR / "f1_threshold_10_percent_vs_sensor_snr_mp_initialization.pdf",
    y_col="f1_threshold_10_percent",
)
